<a href="https://colab.research.google.com/github/yomiannixs-jpg/AI-Optimization-app-using-LSTM_TensorFlow/blob/main/WAR_RISK_AND_CROSS_ASSET_RETURN_CONNECTEDNESS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# WAR RISK AND CROSS-ASSET RETURN CONNECTEDNESS
# Empirical Code V2: Baseline + Robustness + Manuscript Tables
# ============================================================
# This version reconciles the manuscript with the empirical analysis.
# It implements:
#   1. File upload prompt for your merged data
#   2. Baseline Diebold-Yilmaz generalized FEVD connectedness
#   3. AIC/BIC lag-order diagnostics
#   4. Alternative VAR-lag robustness: p = 1, 2, 3
#   5. Alternative FEVD-horizon robustness: H = 10, 20, 50
#   6. Brent oil instead of WTI robustness
#   7. Treasury bonds instead of corporate bonds robustness
#   8. VIX and OVX augmented-system robustness
#   9. Optional GPR quantile regression, if a GPR CSV is supplied
#   10. Auto-download results as output.zip
# ============================================================

import os
import glob
import warnings
import zipfile
import tempfile
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import skew, kurtosis, jarque_bera
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

try:
    import statsmodels.formula.api as smf
except Exception:
    smf = None

# Check if running in Google Colab
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# ============================================================
# 1. FILE UPLOAD PROMPT
# ============================================================

def upload_data_file():
    """Prompt user to upload a CSV file with merged price data."""

    print("=" * 60)
    print("FILE UPLOAD REQUIRED")
    print("=" * 60)
    print("\nPlease upload your merged daily prices CSV file.")
    print("The file should contain the following columns:")
    print("  - Date (format: YYYY-MM-DD)")
    print("  - GC=F (Gold futures)")
    print("  - CL=F (WTI Oil futures)")
    print("  - URTH (MSCI World ETF)")
    print("  - DX-Y.NYB (US Dollar Index)")
    print("  - LQD (Corporate Bonds ETF)")
    print("\nOptional columns for robustness checks:")
    print("  - BZ=F (Brent Oil futures)")
    print("  - TLT (Treasury bonds ETF)")
    print("  - ^VIX (VIX volatility index)")
    print("  - ^OVX (Oil VIX)")
    print("=" * 60)

    if IN_COLAB:
        # Google Colab upload
        uploaded = colab_files.upload()
        if uploaded:
            filepath = list(uploaded.keys())[0]
            print(f"\nFile uploaded: {filepath}")
            return filepath
        else:
            raise FileNotFoundError("No file uploaded. Please run the cell again and select your CSV file.")
    else:
        # Local environment - prompt for file path
        print("\nEnter the path to your CSV file:")
        print("(e.g., merged_daily_prices.csv or /path/to/your/file.csv)")
        filepath = input("File path: ").strip()

        # If user just presses Enter, try default
        if not filepath:
            filepath = "merged_daily_prices.csv"

        # Try to find the file if it doesn't exist at the given path
        if not os.path.exists(filepath):
            print(f"\nFile not found at: {filepath}")
            print("Searching in current directory and subdirectories...")

            # Search for CSV files
            csv_files = glob.glob("**/*.csv", recursive=True)
            csv_files = [f for f in csv_files if "price" in f.lower() or "merged" in f.lower()]

            if csv_files:
                print("\nFound these CSV files:")
                for i, f in enumerate(csv_files):
                    print(f"  {i+1}. {f}")
                print("\nEnter the number of the file to use, or enter 0 to cancel:")
                choice = input("Choice: ").strip()
                try:
                    idx = int(choice) - 1
                    if idx >= 0 and idx < len(csv_files):
                        filepath = csv_files[idx]
                        print(f"\nSelected: {filepath}")
                    else:
                        raise FileNotFoundError("No file selected.")
                except ValueError:
                    raise FileNotFoundError("Invalid selection.")
            else:
                raise FileNotFoundError("No CSV files found. Please place your file in the current directory.")

        return filepath


# ============================================================
# 2. AUTO-DOWNLOAD FUNCTION
# ============================================================

def create_and_download_zip(output_dir, zip_name="output.zip"):
    """
    Create a zip file of all results and download to user's device.
    Works in both Colab and local environments.
    """
    print("\n" + "=" * 60)
    print("PREPARING DOWNLOAD")
    print("=" * 60)

    # Check if output directory exists and has files
    if not os.path.exists(output_dir):
        print(f"Warning: Output directory '{output_dir}' does not exist.")
        return False

    files_list = os.listdir(output_dir)
    if not files_list:
        print(f"Warning: No files found in '{output_dir}' to zip.")
        return False

    # Create zip file
    zip_path = os.path.join(os.getcwd(), zip_name)

    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(output_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, os.path.dirname(output_dir))
                    zipf.write(file_path, arcname)

        file_size = os.path.getsize(zip_path) / (1024 * 1024)  # Size in MB
        print(f"\nZip file created: {zip_name} ({file_size:.2f} MB)")
        print(f"Contains {len(files_list)} files from '{output_dir}'")

        # Download the zip file
        if IN_COLAB:
            # Colab download
            print("\nDownloading to your device...")
            colab_files.download(zip_path)
            print("Download complete!")
        else:
            # Local environment - provide instructions
            print(f"\nZip file saved to: {zip_path}")
            print("You can find it in your current working directory.")
            print("\nTo download in Jupyter Notebook, you can use:")
            print("  from IPython.display import FileLink")
            print(f"  FileLink('{zip_name}')")

            # Try to open file location (Windows/Mac/Linux)
            try:
                import platform
                system = platform.system()
                if system == "Windows":
                    os.startfile(os.path.dirname(zip_path))
                elif system == "Darwin":  # macOS
                    os.system(f"open {os.path.dirname(zip_path)}")
                elif system == "Linux":
                    os.system(f"xdg-open {os.path.dirname(zip_path)}")
            except Exception:
                pass

        return True

    except Exception as e:
        print(f"Error creating zip file: {e}")
        return False


# ============================================================
# 3. USER SETTINGS
# ============================================================

# Get the data file from user
DATA_PATH = upload_data_file()

# Output directory
OUT_DIR = "results_v2"
os.makedirs(OUT_DIR, exist_ok=True)

# Analysis parameters
WAR_START = "2026-03-02"
PREWAR_END = "2026-02-27"
BASE_LAG = 1
BASE_HORIZON = 20
ROLLING_WINDOW = 200

# Baseline variables used in the main manuscript
BASELINE_VARIABLES = {
    "GC=F": "Gold",
    "CL=F": "WTI Oil",
    "URTH": "Global equities",
    "DX-Y.NYB": "US Dollar",
    "LQD": "Corporate bonds",
}

print(f"\nUsing data file: {DATA_PATH}")
print(f"Saving output to: {OUT_DIR}")


# ============================================================
# 4. DATA LOADING
# ============================================================

def load_price_data(filepath):
    """Load and prepare price data from CSV."""
    raw_prices = pd.read_csv(filepath)

    if "Date" not in raw_prices.columns:
        raise ValueError("The data file must contain a Date column.")

    raw_prices["Date"] = pd.to_datetime(raw_prices["Date"])
    raw_prices = raw_prices.set_index("Date").sort_index()

    print(f"\nData loaded: {len(raw_prices)} rows from {raw_prices.index[0].date()} to {raw_prices.index[-1].date()}")
    print(f"Available columns: {', '.join(raw_prices.columns)}")

    return raw_prices

raw_prices = load_price_data(DATA_PATH)


# ============================================================
# 5. CORE DATA FUNCTIONS
# ============================================================

def make_price_panel(price_data, variables):
    """Extract and rename variables for analysis."""
    missing = [c for c in variables if c not in price_data.columns]
    if missing:
        print(f"Warning: Missing required columns: {missing}")
        print("Continuing with available columns only.")
        available_vars = {k: v for k, v in variables.items() if k in price_data.columns}
        if not available_vars:
            raise ValueError("No matching columns found.")
        panel = price_data[list(available_vars.keys())].rename(columns=available_vars).dropna()
    else:
        panel = price_data[list(variables.keys())].rename(columns=variables).dropna()

    print(f"Panel created: {len(panel.columns)} assets, {len(panel)} observations")
    return panel


def make_returns(price_panel):
    """Calculate daily returns in percentage points."""
    return 100 * np.log(price_panel / price_panel.shift(1)).dropna()


def split_sample(returns, war_start=WAR_START):
    """Split returns into pre-war and war periods."""
    ws = pd.to_datetime(war_start)
    pre = returns.loc[returns.index < ws].copy()
    war = returns.loc[returns.index >= ws].copy()

    if len(pre) < 30:
        print(f"Warning: Pre-war sample has only {len(pre)} observations.")
    if len(war) < 30:
        print(f"Warning: War sample has only {len(war)} observations.")

    return pre, war


# ============================================================
# 6. SUMMARY STATISTICS
# ============================================================

def summary_statistics(df):
    """Calculate summary statistics for each series."""
    rows = []
    for col in df.columns:
        x = df[col].dropna()
        jb_stat, jb_p = jarque_bera(x)
        adf_stat, adf_p, *_ = adfuller(x)
        rows.append({
            "Asset": col,
            "Mean": x.mean(),
            "Std. Dev.": x.std(),
            "Variance": x.var(),
            "Skewness": skew(x),
            "Kurtosis": kurtosis(x, fisher=False),
            "Jarque-Bera": jb_stat,
            "JB p-value": jb_p,
            "ADF stat": adf_stat,
            "ADF p-value": adf_p,
            "Obs": len(x),
        })
    return pd.DataFrame(rows)


# ============================================================
# 7. DIEBOLD-YILMAZ GFEVD CONNECTEDNESS
# ============================================================

def generalized_fevd(var_result, horizon=20):
    """
    Compute Generalized Forecast Error Variance Decomposition.
    Pesaran-Shin (1998) GFEVD implementation.
    """
    sigma = np.asarray(var_result.sigma_u)
    k = sigma.shape[0]
    ma_mats = var_result.ma_rep(maxn=horizon)
    fevd = np.zeros((k, k))

    for i in range(k):
        e_i = np.zeros((k, 1))
        e_i[i, 0] = 1.0
        for j in range(k):
            e_j = np.zeros((k, 1))
            e_j[j, 0] = 1.0
            numerator = 0.0
            denominator = 0.0
            for h in range(horizon):
                Ah = ma_mats[h]
                numerator += float((e_i.T @ Ah @ sigma @ e_j) ** 2)
                denominator += float(e_i.T @ Ah @ sigma @ Ah.T @ e_i)
            fevd[i, j] = (1.0 / sigma[j, j]) * numerator / denominator

    # Row normalization
    row_sums = fevd.sum(axis=1, keepdims=True)
    fevd = fevd / row_sums
    return fevd


def connectedness_table(df, lags=1, horizon=20):
    """
    Produce full Diebold-Yilmaz connectedness table.
    Returns table, TCI, net connectedness, and VAR results.
    """
    model = VAR(df)
    result = model.fit(lags)
    fevd = generalized_fevd(result, horizon=horizon)

    names = df.columns
    conn = pd.DataFrame(100.0 * fevd, index=names, columns=names)

    from_others = conn.sum(axis=1) - np.diag(conn)
    to_others = conn.sum(axis=0) - np.diag(conn)
    net = to_others - from_others

    conn["FROM"] = from_others

    to_row = pd.Series(
        list(to_others) + [to_others.sum()],
        index=list(names) + ["FROM"],
        name="TO"
    )
    net_row = pd.Series(
        list(net) + [np.nan],
        index=list(names) + ["FROM"],
        name="NET"
    )

    table = pd.concat([conn, to_row.to_frame().T, net_row.to_frame().T])
    tci = float(to_others.sum() / len(names))

    return table, tci, net, result


# ============================================================
# 8. LAG SELECTION
# ============================================================

def lag_selection_table(df, maxlags=10):
    """Compute AIC, BIC, HQIC, and FPE lag selection criteria."""
    model = VAR(df)
    try:
        maxlags = min(maxlags, max(1, len(df) // 10))
        sel = model.select_order(maxlags=maxlags)
        out = pd.DataFrame({
            "criterion": ["AIC", "BIC", "HQIC", "FPE"],
            "selected_lag": [
                sel.selected_orders.get("aic"),
                sel.selected_orders.get("bic"),
                sel.selected_orders.get("hqic"),
                sel.selected_orders.get("fpe"),
            ],
        })
        return out, sel
    except Exception as e:
        print(f"Lag selection failed: {e}")
        return pd.DataFrame(), None


# ============================================================
# 9. ROLLING TCI
# ============================================================

def rolling_tci(df, window=200, lags=1, horizon=20):
    """Calculate rolling Total Connectedness Index."""
    rows = []
    if len(df) <= window:
        print(f"Warning: Only {len(df)} observations. Need at least {window} for rolling analysis.")
        return pd.DataFrame(columns=["TCI"])

    print(f"\nComputing rolling TCI with {window}-day windows...")
    for i in range(window, len(df)):
        sample = df.iloc[i-window:i]
        try:
            _, tci, _, _ = connectedness_table(sample, lags=lags, horizon=horizon)
            rows.append({"Date": df.index[i], "TCI": tci})
        except Exception as e:
            continue

    if not rows:
        print("Warning: No rolling windows could be computed.")
        return pd.DataFrame(columns=["TCI"])

    print(f"Rolling TCI computed: {len(rows)} windows")
    return pd.DataFrame(rows).set_index("Date")


# ============================================================
# 10. RUN BASELINE ANALYSIS
# ============================================================

print("\n" + "=" * 60)
print("RUNNING BASELINE ANALYSIS")
print("=" * 60)

# Prepare data
prices = make_price_panel(raw_prices, BASELINE_VARIABLES)
returns = make_returns(prices)
prewar, war = split_sample(returns, WAR_START)

# Save data
prices.to_csv(os.path.join(OUT_DIR, "baseline_prices.csv"))
returns.to_csv(os.path.join(OUT_DIR, "baseline_returns.csv"))

# Summary statistics
summary_statistics(returns).to_csv(os.path.join(OUT_DIR, "table1_summary_statistics.csv"), index=False)
returns.corr().to_csv(os.path.join(OUT_DIR, "table2_correlations.csv"))

# Lag diagnostics
for name, df in [("full", returns), ("prewar", prewar), ("war", war)]:
    lags_tbl, _ = lag_selection_table(df, maxlags=min(10, max(1, len(df)//10)))
    if not lags_tbl.empty:
        lags_tbl.to_csv(os.path.join(OUT_DIR, f"lag_selection_{name}.csv"), index=False)

# Connectedness tables
pre_table, tci_pre, net_pre, _ = connectedness_table(prewar, lags=BASE_LAG, horizon=BASE_HORIZON)
war_table, tci_war, net_war, _ = connectedness_table(war, lags=BASE_LAG, horizon=BASE_HORIZON)

pre_table.to_csv(os.path.join(OUT_DIR, "table3_prewar_connectedness.csv"))
war_table.to_csv(os.path.join(OUT_DIR, "table4_war_connectedness.csv"))

# Net connectedness comparison
net_compare = pd.DataFrame({
    "Pre-war": net_pre,
    "War": net_war,
    "Change": net_war - net_pre
})
net_compare.to_csv(os.path.join(OUT_DIR, "table6_net_connectedness_comparison.csv"))

# TCI comparison
tci_compare = pd.DataFrame([
    {"Specification": "Baseline", "Period": "Pre-war", "TCI": tci_pre},
    {"Specification": "Baseline", "Period": "War", "TCI": tci_war},
    {"Specification": "Baseline", "Period": "Absolute increase", "TCI": tci_war - tci_pre},
    {"Specification": "Baseline", "Period": "Relative increase (%)", "TCI": (tci_war - tci_pre) / tci_pre * 100.0},
])
tci_compare.to_csv(os.path.join(OUT_DIR, "table5_tci_comparison.csv"), index=False)

print("\nBaseline results:")
print(f"  Pre-war TCI: {tci_pre:.2f}%")
print(f"  War TCI: {tci_war:.2f}%")
print(f"  Absolute increase: {tci_war - tci_pre:.2f} percentage points")
print(f"  Relative increase: {(tci_war - tci_pre) / tci_pre * 100:.2f}%")


# ============================================================
# 11. ROBUSTNESS: HORIZONS AND LAGS
# ============================================================

print("\n" + "=" * 60)
print("ROBUSTNESS CHECKS: HORIZON AND LAG")
print("=" * 60)

# Horizon robustness
horizon_rows = []
for H in [10, 20, 50]:
    print(f"  Running horizon H={H}...")
    pre_tbl, pre_tci, pre_net, _ = connectedness_table(prewar, lags=BASE_LAG, horizon=H)
    war_tbl, war_tci, war_net, _ = connectedness_table(war, lags=BASE_LAG, horizon=H)
    horizon_rows.append({
        "Horizon": H,
        "Pre-war TCI": pre_tci,
        "War TCI": war_tci,
        "Absolute increase": war_tci - pre_tci,
        "Relative increase (%)": (war_tci - pre_tci) / tci_pre * 100.0,
        "War top transmitter": war_net.idxmax(),
        "War top transmitter NET": war_net.max(),
    })
    pre_tbl.to_csv(os.path.join(OUT_DIR, f"robustness_horizon_{H}_prewar.csv"))
    war_tbl.to_csv(os.path.join(OUT_DIR, f"robustness_horizon_{H}_war.csv"))

pd.DataFrame(horizon_rows).to_csv(os.path.join(OUT_DIR, "table7_horizon_robustness.csv"), index=False)

# Lag robustness
lag_rows = []
for p in [1, 2, 3]:
    print(f"  Running lag p={p}...")
    try:
        pre_tbl, pre_tci, pre_net, _ = connectedness_table(prewar, lags=p, horizon=BASE_HORIZON)
        war_tbl, war_tci, war_net, _ = connectedness_table(war, lags=p, horizon=BASE_HORIZON)
        lag_rows.append({
            "VAR lag": p,
            "Pre-war TCI": pre_tci,
            "War TCI": war_tci,
            "Absolute increase": war_tci - pre_tci,
            "Relative increase (%)": (tci_war - tci_pre) / tci_pre * 100.0,
            "War top transmitter": war_net.idxmax(),
            "War top transmitter NET": war_net.max(),
        })
        pre_tbl.to_csv(os.path.join(OUT_DIR, f"robustness_lag_{p}_prewar.csv"))
        war_tbl.to_csv(os.path.join(OUT_DIR, f"robustness_lag_{p}_war.csv"))
    except Exception as e:
        lag_rows.append({"VAR lag": p, "Error": str(e)})

pd.DataFrame(lag_rows).to_csv(os.path.join(OUT_DIR, "table8_lag_robustness.csv"), index=False)

print("Robustness checks complete.")


# ============================================================
# 12. ROBUSTNESS: ALTERNATIVE ASSET SYSTEMS
# ============================================================

print("\n" + "=" * 60)
print("ROBUSTNESS CHECKS: ALTERNATIVE ASSET SYSTEMS")
print("=" * 60)

def run_system(system_name, variables, lags=BASE_LAG, horizon=BASE_HORIZON):
    """
    Run connectedness analysis for an alternative asset system.
    """
    try:
        p = make_price_panel(raw_prices, variables)
        r = make_returns(p)
        pre, ww = split_sample(r, WAR_START)
        pre_tbl, pre_tci, pre_net, _ = connectedness_table(pre, lags=lags, horizon=horizon)
        war_tbl, war_tci, war_net, _ = connectedness_table(ww, lags=lags, horizon=horizon)
        pre_tbl.to_csv(os.path.join(OUT_DIR, f"{system_name}_prewar_connectedness.csv"))
        war_tbl.to_csv(os.path.join(OUT_DIR, f"{system_name}_war_connectedness.csv"))

        print(f"  {system_name}: Pre-war TCI={pre_tci:.2f}%, War TCI={war_tci:.2f}%")

        return {
            "Specification": system_name,
            "No. assets": len(variables),
            "Pre-war TCI": pre_tci,
            "War TCI": war_tci,
            "Absolute increase": war_tci - pre_tci,
            "Relative increase (%)": (war_tci - pre_tci) / tci_pre * 100.0,
            "War top transmitter": war_net.idxmax(),
            "War top transmitter NET": war_net.max(),
        }
    except Exception as e:
        print(f"  Error in {system_name}: {e}")
        return {
            "Specification": system_name,
            "Error": str(e),
        }

system_rows = []

# Baseline
system_rows.append(run_system("baseline_wti_lqd", BASELINE_VARIABLES))

# Brent oil
if "BZ=F" in raw_prices.columns:
    brent_vars = {
        "GC=F": "Gold",
        "BZ=F": "Brent Oil",
        "URTH": "Global equities",
        "DX-Y.NYB": "US Dollar",
        "LQD": "Corporate bonds",
    }
    system_rows.append(run_system("robustness_brent_lqd", brent_vars))
else:
    print("  Skipping Brent: 'BZ=F' not in data")

# Treasury bonds
if "TLT" in raw_prices.columns:
    tlt_vars = {
        "GC=F": "Gold",
        "CL=F": "WTI Oil",
        "URTH": "Global equities",
        "DX-Y.NYB": "US Dollar",
        "TLT": "Treasury bonds",
    }
    system_rows.append(run_system("robustness_wti_tlt", tlt_vars))
else:
    print("  Skipping TLT: 'TLT' not in data")

# VIX
if "^VIX" in raw_prices.columns:
    vix_vars = dict(BASELINE_VARIABLES)
    vix_vars["^VIX"] = "VIX"
    system_rows.append(run_system("robustness_add_vix", vix_vars))
else:
    print("  Skipping VIX: '^VIX' not in data")

# OVX
if "^OVX" in raw_prices.columns:
    ovx_vars = dict(BASELINE_VARIABLES)
    ovx_vars["^OVX"] = "OVX"
    system_rows.append(run_system("robustness_add_ovx", ovx_vars))
else:
    print("  Skipping OVX: '^OVX' not in data")

# Both VIX and OVX
if "^VIX" in raw_prices.columns and "^OVX" in raw_prices.columns:
    vix_ovx_vars = dict(BASELINE_VARIABLES)
    vix_ovx_vars["^VIX"] = "VIX"
    vix_ovx_vars["^OVX"] = "OVX"
    system_rows.append(run_system("robustness_add_vix_ovx", vix_ovx_vars))

pd.DataFrame(system_rows).to_csv(os.path.join(OUT_DIR, "table9_system_robustness.csv"), index=False)


# ============================================================
# 13. ROLLING TCI AND FIGURES
# ============================================================

print("\n" + "=" * 60)
print("GENERATING FIGURES")
print("=" * 60)

# Rolling TCI
rolling = rolling_tci(returns, window=ROLLING_WINDOW, lags=BASE_LAG, horizon=BASE_HORIZON)
if not rolling.empty:
    rolling.to_csv(os.path.join(OUT_DIR, "rolling_tci.csv"))

# Figure 1: prices
print("  Figure 1: Prices")
axes = prices.plot(subplots=True, figsize=(12, 8), title="Raw asset prices")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure1_prices.png"), dpi=300)
plt.close()

# Figure 2: returns
print("  Figure 2: Returns")
axes = returns.plot(subplots=True, figsize=(12, 8), title="Daily asset returns")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure2_returns.png"), dpi=300)
plt.close()

# Figure 3: rolling TCI
if not rolling.empty:
    print("  Figure 3: Rolling TCI")
    plt.figure(figsize=(12, 5))
    plt.plot(rolling.index, rolling["TCI"], linewidth=1.5)
    plt.axvline(pd.to_datetime(WAR_START), linestyle="--", linewidth=1.0, color='red')
    plt.title("Dynamic Total Connectedness Index")
    plt.ylabel("TCI (%)")
    plt.xlabel("Date")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "figure3_rolling_tci.png"), dpi=300)
    plt.close()

# Figure 4: net connectedness
print("  Figure 4: Net connectedness")
ax = net_compare[["Pre-war", "War"]].plot(kind="bar", figsize=(10, 5))
ax.axhline(0, linewidth=0.8, color='black')
ax.set_title("Net connectedness: pre-war vs war")
ax.set_ylabel("Net connectedness")
ax.set_xlabel("Asset")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure4_net_connectedness.png"), dpi=300)
plt.close()

print("Figures complete.")


# ============================================================
# 14. OPTIONAL QUANTILE REGRESSION WITH GPR
# ============================================================

print("\n" + "=" * 60)
print("OPTIONAL: GPR QUANTILE REGRESSION")
print("=" * 60)

def find_gpr_file():
    """Search for GPR CSV file in various locations."""
    patterns = ["**/*gpr*.csv", "**/*GPR*.csv", "**/*Geopolitical*.csv"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(pat, recursive=True))
    return list(dict.fromkeys(files))

def run_gpr_quantile_regression(rolling_tci, gpr_path):
    """Run quantile regression of TCI on GPR."""
    if smf is None:
        print("  statsmodels formula API not available.")
        return None

    try:
        gpr = pd.read_csv(gpr_path)
        if "Date" not in gpr.columns:
            raise ValueError("GPR file must contain a Date column.")

        gpr["Date"] = pd.to_datetime(gpr["Date"])

        # Find GPR column
        possible = [c for c in gpr.columns if c.lower() in ["gpr", "gpr_index", "geopolitical"]]
        if possible:
            gpr_col = possible[0]
        else:
            numeric_cols = [c for c in gpr.select_dtypes(include=[np.number]).columns if c != "Date"]
            if not numeric_cols:
                raise ValueError("No numeric GPR column found.")
            gpr_col = numeric_cols[0]

        gpr = gpr[["Date", gpr_col]].rename(columns={gpr_col: "GPR"})

        # Merge with TCI
        tci = rolling_tci.reset_index().rename(columns={"index": "Date"})
        data = pd.merge(tci, gpr, on="Date", how="inner").dropna()

        if len(data) < 30:
            print(f"  Only {len(data)} merged observations; skipping.")
            return None

        print(f"  Merged data: {len(data)} observations")

        # Run quantile regression
        rows = []
        for q in np.arange(0.05, 1.00, 0.05):
            try:
                res = smf.quantreg("TCI ~ GPR", data).fit(q=q)
                ci = res.conf_int().loc["GPR"]
                rows.append({
                    "Quantile": q,
                    "Beta_GPR": res.params["GPR"],
                    "Lower_CI": ci[0],
                    "Upper_CI": ci[1],
                    "P_value": res.pvalues["GPR"],
                    "Obs": len(data),
                })
            except Exception as e:
                rows.append({"Quantile": q, "Error": str(e), "Obs": len(data)})

        out = pd.DataFrame(rows)
        out.to_csv(os.path.join(OUT_DIR, "table10_gpr_quantile_regression.csv"), index=False)

        # Create figure
        if "Beta_GPR" in out.columns:
            plt.figure(figsize=(10, 5))
            plt.plot(out["Quantile"], out["Beta_GPR"], marker="o", linewidth=2)
            if "Lower_CI" in out.columns and "Upper_CI" in out.columns:
                plt.fill_between(out["Quantile"], out["Lower_CI"], out["Upper_CI"], alpha=0.2)
            plt.axhline(0, linewidth=0.8, color='black')
            plt.title("Quantile Regression: GPR Effect on Total Connectedness")
            plt.xlabel("Quantile")
            plt.ylabel("GPR Coefficient")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(OUT_DIR, "figure5_gpr_quantile_regression.png"), dpi=300)
            plt.close()
            print("  GPR quantile regression complete.")

        return out

    except Exception as e:
        print(f"  Error in GPR analysis: {e}")
        return None

# Check if GPR file exists and run if available
if not rolling.empty:
    gpr_files = find_gpr_file()
    if gpr_files:
        print(f"  Found GPR file: {os.path.basename(gpr_files[0])}")
        run_gpr_quantile_regression(rolling, gpr_files[0])
    else:
        print("  No GPR CSV file found. To run quantile regression,")
        print("  add a CSV file with 'Date' and 'GPR' columns to the working directory.")
        with open(os.path.join(OUT_DIR, "GPR_not_found.txt"), "w") as f:
            f.write("No GPR file found in the working directory.\n")
else:
    print("  Skipping GPR analysis: Rolling TCI not available.")


# ============================================================
# 15. SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)

summary_lines = [
    "WAR RISK AND CROSS-ASSET CONNECTEDNESS: EMPIRICAL V2 SUMMARY",
    "=" * 60,
    f"Data file: {os.path.basename(DATA_PATH)}",
    f"Sample period: {returns.index[0].date()} to {returns.index[-1].date()}",
    f"Pre-war observations: {len(prewar)}",
    f"War observations: {len(war)}",
    f"Baseline lag: VAR({BASE_LAG})",
    f"Baseline FEVD horizon: H={BASE_HORIZON}",
    f"Rolling window: W={ROLLING_WINDOW}",
    "",
    "BASELINE RESULTS:",
    f"  Pre-war TCI: {tci_pre:.2f}%",
    f"  War TCI: {tci_war:.2f}%",
    f"  Absolute increase: {tci_war - tci_pre:.2f} percentage points",
    f"  Relative increase: {(tci_war - tci_pre) / tci_pre * 100:.2f}%",
    f"War-period top net transmitter: {net_war.idxmax()} ({net_war.max():.2f})",
    f"War-period strongest net receiver: {net_war.idxmin()} ({net_war.min():.2f})",
    "",
    "GENERATED FILES:",
]

# List all generated files
files_list = os.listdir(OUT_DIR)
if files_list:
    for f in sorted(files_list):
        file_size = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
        summary_lines.append(f"  {f} ({file_size:.2f} KB)")
else:
    summary_lines.append("  No files generated.")

summary_lines.append("")
summary_lines.append("=" * 60)

# Save summary
with open(os.path.join(OUT_DIR, "analysis_summary.txt"), "w") as f:
    f.write("\n".join(summary_lines))

# Print summary
print("\n".join(summary_lines))

print(f"\nAll outputs saved in: {OUT_DIR}/")

# Attempt to auto-download results zip
create_and_download_zip(OUT_DIR)


FILE UPLOAD REQUIRED

Please upload your merged daily prices CSV file.
The file should contain the following columns:
  - Date (format: YYYY-MM-DD)
  - GC=F (Gold futures)
  - CL=F (WTI Oil futures)
  - URTH (MSCI World ETF)
  - DX-Y.NYB (US Dollar Index)
  - LQD (Corporate Bonds ETF)

Optional columns for robustness checks:
  - BZ=F (Brent Oil futures)
  - TLT (Treasury bonds ETF)
  - ^VIX (VIX volatility index)
  - ^OVX (Oil VIX)


Saving merged_daily_prices.csv to merged_daily_prices (3).csv

File uploaded: merged_daily_prices (3).csv

Using data file: merged_daily_prices (3).csv
Saving output to: results_v2

Data loaded: 1625 rows from 2020-01-02 to 2026-06-22
Available columns: ^OVX, ^VIX, BZ=F, CL=F, DX-Y.NYB, GC=F, LQD, TLT, URTH

RUNNING BASELINE ANALYSIS
Panel created: 5 assets, 1625 observations

Baseline results:
  Pre-war TCI: 19.86%
  War TCI: 50.83%
  Absolute increase: 30.97 percentage points
  Relative increase: 155.94%

ROBUSTNESS CHECKS: HORIZON AND LAG
  Running horizon H=10...
  Running horizon H=20...
  Running horizon H=50...
  Running lag p=1...
  Running lag p=2...
  Running lag p=3...
Robustness checks complete.

ROBUSTNESS CHECKS: ALTERNATIVE ASSET SYSTEMS
Panel created: 5 assets, 1625 observations
  baseline_wti_lqd: Pre-war TCI=19.86%, War TCI=50.83%
Panel created: 5 assets, 1625 observations
  robustness_brent_lqd: Pre-war TCI=20.52%, War TCI=51.50%
Panel created: 5 assets, 1625 observa

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete!


True